In [283]:
from sage.all import *
import sage.libs.lrcalc.lrcalc as lrcalc

In [284]:
def degree(partition):
    """Return the degree (sum of parts) of a partition."""
    return sum(partition)

def compare_by_degree(p1, p2):
    """Compare two partitions by their degrees."""
    d1, d2 = degree(p1), degree(p2)
    if d1 < d2:
        return -1
    elif d1 > d2:
        return 1
    return 0


In [285]:

def generate_partitions(n, max_part=None):
    """Generate all partitions of n."""
    if n == 0:
        yield ()
    else:
        if max_part is None or max_part > n:
            max_part = n
        for first in range(max_part, 0, -1):
            for rest in generate_partitions(n - first, first):
                yield (first,) + rest

# --- Example usage ---
p = (3, 1)  # partition

print(p)



(3, 1)


In [286]:
def subset_partitions(partition):
    """Return all partitions with degree <= degree(partition)."""
    d = degree(partition)
    result = []
    for n in range(d + 1):
        result.extend(generate_partitions(n))
    return result

In [287]:

print("Degree of p:", degree(p))

q = (1, 1)
print("Compare p and q:", compare_by_degree(p, q))  # 0 means equal

subset = subset_partitions(p)
print(f"All partitions with degree ≤ {degree(p)}:")
print(subset)


Degree of p: 4
Compare p and q: 1
All partitions with degree ≤ 4:
[(), (1,), (2,), (1, 1), (3,), (2, 1), (1, 1, 1), (4,), (3, 1), (2, 2), (2, 1, 1), (1, 1, 1, 1)]


In [288]:
def solveOne(k, k1, k2, k3, k4):
    """
    Solve the system

        2*x2 + x1 + x3 + x4 = k                (1)
        k1 - k2 = x2 + x1 + x3                  (2)
        k3 - k4 = x2 + x1 + x4                  (3)

    together with the bounds

        x1,x2,x3,x4 < k1   and   x1,x2,x3,x4 < k3
        k2 < k1 ,   k4 < k3   (these are assumed true, but are checked)

    Parameters
    ----------
    k  : int   – the constant that appears in (1)
    k1 : int   – |λ|
    k2 : int   – |λ'| (already chosen)
    k3 : int   – |μ|
    k4 : int   – |μ'| (already chosen)

    Returns
    -------
    list of dicts, each dict containing the six numbers
    {"deg δ":x1, "deg γ":x2, "p":x3, "q":x4,
     "deg λ'":k2, "deg μ'":k4}
    """
    # -----------------------------------------------------------------
    # 0.  sanity checks that are required by the statement
    # -----------------------------------------------------------------
    if not (k2 < k1 and k4 < k3):
        # the user supplied inadmissible degrees – nothing to do
        return []

    # -----------------------------------------------------------------
    # 1.  Derive x1 directly from (1) + (2) + (3)
    # -----------------------------------------------------------------
    # Adding (2) and (3) and subtracting (1) eliminates x2 completely:
    #   (k1 - k2) + (k3 - k4) - k = x1
    x1 = k1 - k2 + k3 - k4 - k
    if x1 < 0:                     # x1 must be non‑negative
        return []

    # -----------------------------------------------------------------
    # 2.  The remaining unknown is x2 ; once we know x2 we get x3,x4
    # -----------------------------------------------------------------
    # From (2)   :  x3 = (k1 - k2) - (x1 + x2)
    # From (3)   :  x4 = (k3 - k4) - (x1 + x2)
    # Both x3 and x4 must be ≥ 0, therefore
    #   x1 + x2 ≤ k1 - k2   and   x1 + x2 ≤ k3 - k4
    max_sum = min(k1 - k2, k3 - k4)      # the largest allowed value of (x1+x2)

    # The smallest possible sum is just x1 (when x2 = 0)
    if x1 > max_sum:                     # no room for a non‑negative x2
        return []

    solutions = []
    # x2 can range from 0 up to the value that keeps the sum ≤ max_sum
    max_x2 = max_sum - x1
    for x2 in range(max_x2 + 1):         # inclusive upper bound
        s = x1 + x2                       # s = x1 + x2

        x3 = (k1 - k2) - s
        x4 = (k3 - k4) - s

        # all variables must be non‑negative
        if x3 < 0 or x4 < 0:
            continue

        # condition (4): each of x1,x2,x3,x4 < k1 and < k3
        if not (x1 < k1 and x2 < k1 and x3 < k1 and x4 < k1):
            continue
        if not (x1 < k3 and x2 < k3 and x3 < k3 and x4 < k3):
            continue

        # everything checks out – store the solution
        solutions.append({
            "deg δ": x1,
            "deg γ": x2,
            "p":    x3,
            "q":    x4,
            "deg λ'": k2,
            "deg μ'": k4
        })

    return solutions

In [289]:
def solveTwo(mu1, mu2, mu3, mu4):

    v1 = mu1 + mu2
    v2 = mu3 + mu4
    

    return [v1, v2]


In [290]:
# -----------------------------------------------------------------
#  three‑fold LR convolution – unchanged logic, corrected arguments
# -----------------------------------------------------------------
def lrcoef4(mu1, mu2, mu3, mu4, lam):
    """
    Return
        Σ_{a ⊢ (mu1+mu2)} Σ_{b ⊢ (mu3+mu4)}
               c^{a}_{mu1,mu2} · c^{b}_{mu3,mu4} · c^{lam}_{a,b}

    The first four arguments are *integers* (the degrees that appear in the
    linear system).  ``lam`` must be the *partition* λ (a tuple), not its
    total degree.
    """
    total1 = mu1 + mu2                # degree of the first intermediate partition
    total2 = mu3 + mu4                # degree of the second intermediate partition

    parts_a = list(generate_partitions(total1))
    parts_b = list(generate_partitions(total2))

    s = 0
    for a in parts_a:
        c1 = int(lrcalc.lrcoef((mu1,), (mu2,), a))   # c^{a}_{mu1,mu2}
        if c1 == 0:
            continue
        for b in parts_b:
            c2 = int(lrcalc.lrcoeff((mu3,), (mu4,), b))   # c^{b}_{mu3,mu4}
            if c2 == 0:
                continue
            c3 = int(lrcalc.lrcoeff(a, b, lam))           # c^{lam}_{a,b}
            if c3 == 0:
                continue
            s += c1 * c2 * c3
    return s


In [291]:
def CalcSoc(k, k1, k2, k3, k4, lam, mu):
    """
    k1 = |lam|,   k3 = |mu|
    """
    L = solveOne(k, k1, k2, k3, k4)

    tot_sum = 0
    for sol in L:
        δ = sol["deg δ"]
        γ = sol["deg γ"]
        p = sol["p"]
        q = sol["q"]

        # first factor uses the *full* partition lam
        term1 = lrcoef4(δ, γ, p, k2, lam)

        # second factor uses the *full* partition mu
        term2 = lrcoef4(δ, γ, q, k4, mu)

        tot_sum += term1 * term2
    return tot_sum


In [292]:
def Master1(lam, lamP, mu, muP, k):
    SolveSoc